In [69]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

# --- 辅助函数：计算临床评价指标 ---
def get_clinical_metrics(y_true, y_pred):
    """
    计算并返回医学常用的评价指标：灵敏度(Sensitivity)与特异度(Specificity)
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)  # 召回率：真实患者中被检出的比例
    specificity = tn / (tn + fp)  # 真阴性率：健康人群中被排除的比例
    return sensitivity, specificity

# --- (1) 数据读取 ---
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", 
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]

# 读取数据，并将 '?' 识别为缺失值
df = pd.read_csv(url, names=columns, na_values='?')

# --- 数据清洗 ---
# 删除含缺失值的样本 (仅6例，不影响整体分布)
df.dropna(inplace=True)

# --- (2) 标签二值化与特征标准化 ---
# 标签：>0 为患病(1), =0 为健康(0)
df["target"] = (df["target"] > 0).astype(int)

# 特征：分离并标准化
X_raw = df.drop("target", axis=1)
y = df["target"]

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X_raw), columns=X_raw.columns)

print(f"数据预处理完成。有效样本数: {len(df)}")

数据预处理完成。有效样本数: 297


In [70]:
# --- (1) 划分独立测试集 ---
# 30% 数据作为测试集，全程锁定，仅用于最终评估
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# --- (2) 模拟半监督划分 ---
# 在剩余训练数据中：
# - 有标签集 (Labeled): 仅保留 20%
# - 无标签集 (Unlabeled): 其余 80%
X_labeled, X_unlabeled, y_labeled, _ = train_test_split(
    X_train_full, y_train_full,
    train_size=0.2,  # 模拟仅有 20% 的专家标注
    random_state=42,
    stratify=y_train_full
)

# 重置索引 (关键步骤，防止后续合并时错位)
X_labeled = X_labeled.reset_index(drop=True)
y_labeled = y_labeled.reset_index(drop=True)
X_unlabeled = X_unlabeled.reset_index(drop=True)

print(f"有标签样本数: {len(X_labeled)}")
print(f"无标签样本数: {len(X_unlabeled)}")

有标签样本数: 41
无标签样本数: 166


In [71]:
# 初始化并训练基线模型 (Teacher)
teacher_model = LogisticRegression(max_iter=1000, random_state=42)
teacher_model.fit(X_labeled, y_labeled)

# 评估基线性能
y_pred_base = teacher_model.predict(X_test)
y_prob_base = teacher_model.predict_proba(X_test)[:, 1]

# 计算临床指标
sens_base, spec_base = get_clinical_metrics(y_test, y_pred_base)

print("\n=== 基线模型 (Teacher) ===")
print(f"Accuracy:    {accuracy_score(y_test, y_pred_base):.4f}")
print(f"AUC:         {roc_auc_score(y_test, y_prob_base):.4f}")
print(f"Sensitivity: {sens_base:.4f} (灵敏度)")
print(f"Specificity: {spec_base:.4f} (特异度)")


=== 基线模型 (Teacher) ===
Accuracy:    0.7333
AUC:         0.8636
Sensitivity: 0.6429 (灵敏度)
Specificity: 0.8125 (特异度)


In [72]:
# --- (1) 预测: 获取软概率 ---
# predict_proba 返回形状 (n_samples, 2)
probs_unlabeled = teacher_model.predict_proba(X_unlabeled)

# --- (2) 筛选: 基于置信度 ---
# 获取每个样本的最大预测概率 (即置信度)
confidence = probs_unlabeled.max(axis=1)

# 获取预测的类别 (0 或 1)
predicted_classes = probs_unlabeled.argmax(axis=1)

# 设定阈值 0.85 (保证伪标签的高质量)
threshold = 0.85
high_conf_idx = np.where(confidence > threshold)[0]

# --- (3) 硬化: 提取对应的硬标签 ---
# 直接使用 0/1 类别作为新标签
pseudo_labels = predicted_classes[high_conf_idx]

# 同时提取置信度，用于后续加权
pseudo_weights = confidence[high_conf_idx]

print("\n=== 伪标签生成统计 ===")
print(f"置信度阈值: {threshold}")
print(f"筛选出的高置信样本数: {len(high_conf_idx)}")
print(f"筛选比例: {len(high_conf_idx)/len(X_unlabeled):.1%}")


=== 伪标签生成统计 ===
置信度阈值: 0.85
筛选出的高置信样本数: 80
筛选比例: 48.2%


In [73]:
# 1. 提取筛选出的伪标签数据 (特征 + 标签)
X_pseudo = X_unlabeled.iloc[high_conf_idx]
y_pseudo = pd.Series(pseudo_labels, name="target")

# 2. 数据合并 (Data Augmentation)
# 新训练集 = 原始有标签数据 + 筛选出的伪标签数据
X_new = pd.concat([X_labeled, X_pseudo], axis=0)
y_new = pd.concat([y_labeled, y_pseudo], axis=0)

# 3. 构造样本权重 (Sample Weights)
# - 真实标签样本：权重固定为 1.0 (完全信任)
# - 伪标签样本：权重 = 置信度 (如 0.92, 0.88... 给予部分信任)
weights_labeled = np.ones(len(X_labeled))
sample_weights = np.concatenate([weights_labeled, pseudo_weights])

# 4. 训练 Student 模型
# 关键步骤：传入 sample_weight 参数
student_model = LogisticRegression(max_iter=1000, random_state=42)
student_model.fit(X_new, y_new, sample_weight=sample_weights)

# 5. 最终评估
y_pred_student = student_model.predict(X_test)
y_prob_student = student_model.predict_proba(X_test)[:, 1]

# 计算临床指标
sens_student, spec_student = get_clinical_metrics(y_test, y_pred_student)
acc_student = accuracy_score(y_test, y_pred_student)
auc_student = roc_auc_score(y_test, y_prob_student)

print("\n=== 伪标签模型 (Student) 最终表现 ===")
print(f"Accuracy:    {acc_student:.4f}")
print(f"AUC:         {auc_student:.4f}")
print(f"Sensitivity: {sens_student:.4f} (灵敏度)")
print(f"Specificity: {spec_student:.4f} (特异度)")


=== 伪标签模型 (Student) 最终表现 ===
Accuracy:    0.7556
AUC:         0.8700
Sensitivity: 0.7143 (灵敏度)
Specificity: 0.7917 (特异度)
